# Vermind Pipeline Debug Notebook

This notebook shows the pipeline inputs, the web search results, the final pipeline verdict, and the direct judge response for a single example question.

Use it to inspect missing environment variables and verify the safe fallback behavior when API keys are missing or invalid.

## Configuration inspection

Define the sample question and answer, then display relevant environment configuration.

In [ ]:
import os
from pprint import pprint

question = 'Is Paris the capital of France?'
answer = 'Yes, Paris is the capital of France.'

print('Environment variables:')
for key in ['GROQ_API_KEY', 'GROQ_MODEL', 'WEB_SEARCH_API_KEY', 'WEB_SEARCH_API_URL']:
    value = os.getenv(key)
    if value is None:
        print(f'{key} = <missing>')
    else:
        display_value = f'{value[:20]}...' if len(value) > 20 else value
        print(f'{key} = {display_value}')


## Web search and pipeline

Inspect retrieved web evidence and the result returned by the complete pipeline.

In [ ]:
from web_search import search_web
from pipeline import run

results = search_web(question, k=5)
print(f'search_web() returned {len(results)} result(s)')
if results:
    pprint(results[:3])
else:
    print('No web results available.')

pipeline_result = run(question, answer)
print('\nPipeline result:')
pprint(pipeline_result)

## Direct judge check

Run the evidence-based judge directly using the retrieved evidence.

In [ ]:
from judge import verify_answer

evidence = [
    {
        'doc_id': result.get('url') or result.get('title') or 'web-result',
        'text': f"{result.get('title', '')}\n{result.get('snippet', '')}"
    }
    for result in results
]
if not evidence:
    evidence = [{'doc_id': 'no-web-results', 'text': 'No live web results available.'}]

direct_result = verify_answer(question, answer, evidence)
print('\nDirect judge result:')
pprint(direct_result)

## Optional data inspection

Inspect local benchmark and corpus JSONL files when they are present.

In [ ]:
import json
from pathlib import Path

paths = [
    Path('data/benchmarks/fever.jsonl'),
    Path('data/benchmarks/truthful_qa.jsonl'),
    Path('data/corpus/passages.jsonl'),
]

for path in paths:
    print(f'\n=== {path} ===')
    if not path.exists():
        print('MISSING')
        continue

    with path.open('r', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print('row_count =', len(rows))
    if len(rows) == 0:
        print('empty dataset file')
        continue

    if len(rows) <= 20:
        print('all_rows =')
        for row in rows:
            print(row)
    else:
        print('showing_first_20_rows_only =')
        for row in rows[:20]:
            print(row)
